In [1]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import cartopy.crs as ccrs
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
from shapely.ops import transform
import xarray as xr
import numpy as np
import tarfile


In [2]:
# toggle
TC = 'TC.2'
run = '002'

In [3]:
## read in csv files which compiles data and takes third entry from each individual DN file
# nn mode (negative ENSO (El Nino), negative North Pacific mode)
ds_nn = pd.read_csv(f"datasets/ALCC/post_processing/ALCC_output/{TC}/{run}/{TC}_nn.{run}/{TC}_nn.{run}_SN.csv")
ds_nn.columns = ds_nn.columns.str.strip()
ds_nn['mode'] = 'nn'

# no mode (negative ENSO (El Nino), neutral North Pacific mode)
ds_no = pd.read_csv(f"datasets/ALCC/post_processing/ALCC_output/{TC}/{run}/{TC}_no.{run}/{TC}_no.{run}_SN.csv")
ds_no.columns = ds_no.columns.str.strip()
ds_no['mode'] = 'no'

# np mode (negative ENSO (El Nino), positive North Pacific mode)
ds_np = pd.read_csv(f"datasets/ALCC/post_processing/ALCC_output/{TC}/{run}/{TC}_np.{run}/{TC}_np.{run}_SN.csv")
ds_np.columns = ds_np.columns.str.strip()
ds_np['mode'] = 'np'

# on mode (neutral ENSO, negative North Pacific mode)
ds_on = pd.read_csv(f"datasets/ALCC/post_processing/ALCC_output/{TC}/{run}/{TC}_on.{run}/{TC}_on.{run}_SN.csv")
ds_on.columns = ds_on.columns.str.strip()
ds_on['mode'] = 'on'

# oo mode (neutral ENSO, neutral North Pacific mode)
ds_oo = pd.read_csv(f"datasets/ALCC/post_processing/ALCC_output/{TC}/{run}/{TC}_oo.{run}/{TC}_oo.{run}_SN.csv")
ds_oo.columns = ds_oo.columns.str.strip()
ds_oo['mode'] = 'oo'

# op mode (neutral ENSO, positive North Pacific mode)
ds_op = pd.read_csv(f"datasets/ALCC/post_processing/ALCC_output/{TC}/{run}/{TC}_op.{run}/{TC}_op.{run}_SN.csv")
ds_op.columns = ds_op.columns.str.strip()
ds_op['mode'] = 'op'

# pn mode (positive ENSO (La Nina), negative North Pacific mode)
ds_pn = pd.read_csv(f"datasets/ALCC/post_processing/ALCC_output/{TC}/{run}/{TC}_pn.{run}/{TC}_pn.{run}_SN.csv")
ds_pn.columns = ds_pn.columns.str.strip()
ds_pn['mode'] = 'pn'

# po mode (positive ENSO (La Nina), neutral North Pacific mode)
ds_po = pd.read_csv(f"datasets/ALCC/post_processing/ALCC_output/{TC}/{run}/{TC}_po.{run}/{TC}_po.{run}_SN.csv")
ds_po.columns = ds_po.columns.str.strip()
ds_po['mode'] = 'po'

# pp mode (positive ENSO (La Nina), positive North Pacific mode)
ds_pp = pd.read_csv(f"datasets/ALCC/post_processing/ALCC_output/{TC}/{run}/{TC}_pp.{run}/{TC}_pp.{run}_SN.csv")
ds_pp.columns = ds_pp.columns.str.strip()
ds_pp['mode'] = 'pp'

In [4]:
# select mode to easily toggle between datasets
mode = 'pp'

In [5]:
# create a dictionary for the mode datasets
ds_dict = {
    'nn' : ds_nn,
    'no' : ds_no,
    'np' : ds_np,
    'on' : ds_on,
    'oo' : ds_oo,
    'op' : ds_op,
    'pn' : ds_pn,
    'po' : ds_po,
    'pp' : ds_pp,
}
ds = ds_dict[mode]

In [6]:
# convert lon to 180 scale
ds['lon_180'] = ((ds['lon'] + 180) % 360) - 180

print(ds.head())

   track_id  year  month  day  hour      i         lon       lat       slp  \
0         0  1984      7    6     6  27869  147.417735  7.223966  100459.4   
1         0  1984      7    6    12  27866  146.840600  6.648283  100323.5   
2         0  1984      7    7     0  27889  145.745634  6.737920  100256.2   
3         0  1984      7    7     6  27889  145.745634  6.737920  100092.4   
4         0  1984      7    7    12  27888  145.001326  7.688409  100237.2   

       wind  phis mode     lon_180  
0  15.90421   0.0   pp  147.417735  
1  18.60843   0.0   pp  146.840600  
2  19.24858   0.0   pp  145.745634  
3  19.48328   0.0   pp  145.745634  
4  21.00596   0.0   pp  145.001326  


In [7]:
# read in basin definition file
polygons_dict = {}

# read in basin definition file
with open("tc_basins_NAtl.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",")
        basin_name = parts[0].replace('"', '')
        n_vertices = int(parts[1])

        lon_vals = list(map(float, parts[2:2+n_vertices]))
        lat_vals = list(map(float, parts[2+n_vertices:2+2*n_vertices]))

        coords = list(zip(lon_vals, lat_vals))
        poly = Polygon(coords)

        if basin_name not in polygons_dict:
            polygons_dict[basin_name] = []
        polygons_dict[basin_name].append(poly)

# Convert to GeoDataFrame
basin_records = []

for name, poly_list in polygons_dict.items():
    if len(poly_list) == 1:
        geom = poly_list[0]
    else:
        geom = MultiPolygon(poly_list)

    basin_records.append({
        "basin name": name,
        "geometry": geom
    })

basins = gpd.GeoDataFrame(basin_records, crs="EPSG:4326")

# fix invalid polygons
basins["geometry"] = basins["geometry"].buffer(0)

# remove empy geometries
basins = basins[~basins.geometry.is_empty]

# convert basins' lon to -180-180
basins["geometry"] = basins["geometry"].apply(
    lambda geom: transform(
        lambda x, y: (((x + 180) % 360) - 180, y),
        geom
    )
)

# read in NAtl subbasin polygons
sub_polygons_dict = {}

with open("tc_subbasins_NAtl_v5.dat", "r") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = line.split(",")
        sub_basin_name = parts[0].replace('"', '')
        n_vertices = int(parts[1])

        lon_vals = list(map(float, parts[2:2+n_vertices]))
        lon_vals = [(lon + 180) % 360 - 180 for lon in lon_vals]
        lat_vals = list(map(float, parts[2+n_vertices:2+2*n_vertices]))

        coords = list(zip(lon_vals, lat_vals))
        poly = Polygon(coords)

        if sub_basin_name not in sub_polygons_dict:
            sub_polygons_dict[sub_basin_name] = []
        sub_polygons_dict[sub_basin_name].append(poly)

# Convert to GeoDataFrame
sub_basin_records = []

for name, poly_list in sub_polygons_dict.items():
    if len(poly_list) == 1:
        geom = poly_list[0]
    else:
        geom = MultiPolygon(poly_list)

    sub_basin_records.append({
        "sub_basin_name": name,
        "geometry": geom
    })

sub_basins = gpd.GeoDataFrame(sub_basin_records, crs="EPSG:4326",geometry="geometry")

# fix invalid polygons
sub_basins["geometry"] = sub_basins["geometry"].buffer(0)

# remove empty geometries
sub_basins = sub_basins[~sub_basins.geometry.is_empty]

# longitude conversion
import shapely.ops
def shift_lon(geom):
    return shapely.ops.transform(
        lambda x, y: (((x + 180) % 360) - 180, y),
        geom
    )

# shift lon
sub_basins["geometry"] = sub_basins["geometry"].apply(shift_lon)
basins["geometry"] = basins["geometry"].apply(shift_lon)

In [8]:
# convert LAT and LON to a new column Points which contains (lon, lat) and convert to a geo data frame so we can filter using polygons
ds_points = gpd.GeoDataFrame(
    ds, 
    geometry = gpd.points_from_xy(ds.lon_180, ds.lat),
    crs = "EPSG:4326"
)

# filter points to North Atlantic
ds_filt = gpd.sjoin(
     ds_points,
     basins[basins["basin name"] == "N Atlantic"],
     how = "inner",
     predicate = "within"
)

# filter to only columns we need
ds_filt = ds_filt[['track_id', 'year', 'lon_180', 'lat', 'mode', 'geometry', 'slp', 'wind']]

# join sub basin name for starting and ending points
ds_gdf = gpd.GeoDataFrame(
     ds_filt,
     geometry=gpd.points_from_xy(
         ds_filt.lon_180,
         ds_filt.lat
     ),
     crs=sub_basins.crs
)

ds_join = gpd.sjoin(
     ds_gdf,
     sub_basins[['sub_basin_name', 'geometry']],
     how='left',
     predicate='within'
)

ds_filt['sub_basin_start'] = ds_join['sub_basin_name']

df = ds_filt.copy()

# create 5deg bins
df["lat_bin"] = np.floor(df["lat"] / 5) * 5
df["lon_bin"] = np.floor(df["lon_180"] / 5) * 5

# count nodes in each bin
counts = (
    df.groupby(
        ["year", "sub_basin_start", "mode", "lat_bin", "lon_bin"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "count"})
)

# # save table
# ds_filt.to_csv(f"datasets/ALCC/post_python_processing/{TC}/{run}/density/ALCC_{TC}_{run}_output_density_perYr_wSubbasin_{mode}")
